# Lesson 9: Feature Engineering & Preprocessing

Before a model sees your data, you often need to **clean it** and **shape it** so the model can learn well.

This lesson covers:
1. **Feature engineering** — creating useful inputs from raw data
2. **Missing values** — gaps in the data
3. **Categorical encoding** — turning words/categories into numbers
4. **Feature scaling** — putting numbers on a fair scale
5. **The golden rule** — fit preprocessors on training data only (avoid data leakage)

> Upload to [Google Colab](https://colab.research.google.com) and run top to bottom.

## Step 1: A tiny messy dataset (like real life)

We have: house **size**, **bedrooms**, **city** (text), and **price** (label). Some sizes are missing.

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    'size_sqft': [1500, 2000, np.nan, 1200, 2500, 1800],
    'bedrooms':  [3, 4, 2, 2, 5, 3],
    'city':      ['NYC', 'LA', 'NYC', 'Chicago', 'LA', 'Chicago'],
    'price':     [300000, 450000, 210000, 175000, 540000, 360000]
})
print(df)

## Step 2: Handle missing values

**Missing value** = a blank cell (NaN). Models cannot use blanks.

Common strategies:
- **Drop** rows with missing data (only if few are missing)
- **Impute** = fill with a sensible value (median is common for numbers)

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
df['size_sqft'] = imputer.fit_transform(df[['size_sqft']])
print('After imputing missing size with median:')
print(df)

## Step 3: Feature engineering — create a new useful column

**Feature engineering** = using domain knowledge to build new inputs that help the model.

Example: **price per sqft** often captures value better than raw size alone.

In [ ]:
df['price_per_sqft'] = df['price'] / df['size_sqft']
print(df[['size_sqft', 'price', 'price_per_sqft']])

## Step 4: Encode categories (city names -> numbers)

Models need **numbers**, not text like 'NYC'. **Encoding** converts categories to numeric form.

- **One-hot encoding**: each city gets its own 0/1 column (safest default for most models)
- **Label encoding**: NYC=0, LA=1, ... (only for tree models sometimes; risky for linear models)

In [ ]:
df_encoded = pd.get_dummies(df, columns=['city'], prefix='city')
print('One-hot encoded cities:')
print(df_encoded.head())

## Step 5: Feature scaling — why and when

**Feature scaling** = rescale numeric columns so they are comparable.

Why? Linear models, PCA, K-Means use **distance** or **weighted sums**. If one feature is 0–5000 and another is 0–5, the big one dominates.

Trees do NOT need scaling (yes/no splits only).

Common methods:
- **StandardScaler**: mean=0, std=1 (most common)
- **MinMaxScaler**: squeeze into 0–1 range

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['size_sqft', 'bedrooms', 'price_per_sqft']
scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])
print('Before scaling (first row):')
print(df_encoded[numeric_cols].iloc[0].values)
print('After StandardScaler (first row):')
print(df_scaled[numeric_cols].iloc[0].values)

## Step 6: The golden rule — no data leakage

**Data leakage** = accidentally letting information from the test set influence training (e.g. computing median on ALL data including test).

Rule: **fit** preprocessors (imputer, scaler) on **training data only**, then **transform** train and test.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Rebuild clean feature matrix
features = df_encoded.drop(columns=['price'])
target = df_encoded['price']

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.33, random_state=42)

num_features = ['size_sqft', 'bedrooms', 'price_per_sqft']
preprocess = ColumnTransformer([
    ('scale', StandardScaler(), num_features)
])

pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LinearRegression())
])

pipe.fit(X_train, y_train)
print(f'Train R2: {pipe.score(X_train, y_train):.3f}')
print(f'Test R2:  {pipe.score(X_test, y_test):.3f}')

The **Pipeline** chains preprocessing + model so scaling is learned only from training data automatically.

## Your turn

1. What is **imputation**, and why can't we leave NaN in the data?
2. Why is **one-hot encoding** safer than label encoding for linear regression?
3. Name two model types that need scaling and one that does not.
4. What is **data leakage**, and how does a Pipeline help prevent it?